<a href="https://colab.research.google.com/github/TomazDrumond/Case-Valor/blob/master/Tom_Research_Question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TomazDrumond/Tom_Fly/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm choosing it because my NB1/NB2 work already built the exact building blocks this lane calls for — a stale_visible_page-style hand rule and a readable depth-2 tree — and the guide's own starter results show a learned ranking clearly beats the fixed rule here (baseline Precision@50 = 0.240 vs random forest 0.740), so there's real room to improve on a transparent baseline over the next 7 weeks.

In [6]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(df.shape[0], "pages | declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages | declining rate: 0.542


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Which pages a content/SEO reviewer looks at first this cycle, out of a much
larger inventory than they have time to check manually. RThe reviewer decides to refresh, expand, protect, prune, or monitor the flagged page.<br>
If a false positive spends limited review time on a page that wasn't
actually a priority; a false negative lets a real stale/declining/low-CTR page keep losing visibility unreviewed.
With thousands of pages and one reviewer's limited daily capacity, a ranked queue with reason codes gets the right pages in front of a human far faster and more consistently than manual triage — this is decision-support, not automation.

In [7]:
# Capacity framing: how many pages even qualify as review candidates under the guide's
# own stale_visible_page reason code
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print("stale_visible_page candidates:", stale_visible, "/", df.shape[0])

stale_visible_page candidates: 17 / 30000


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers back this lane, straight from the starter dataset and the guide's verified
starter results:
1. 54.2% of pages are currently flagged as declining — a large enough base rate to matter.
2. 17 pages qualify as stale_visible_page candidates
   right now — more than any reviewer could check by hand each cycle.
3. Per the guide's verified starter results (outputs/model_results.json), a random forest
   reaches Precision@50 = 0.740 versus the baseline rule's 0.240 — meaning about 37 of the
   top 50 flagged pages are real positives with a learned model, versus only 12 with the
   fixed rule. That gap is the room this lane exists to close.

In [8]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
hand_rule_score = stale * visible * df["impressions_90d"]
y = df["is_declining_label"].values

print("Hand rule Precision@20:", round(precision_at_k(hand_rule_score, y, 20), 3))
print("Hand rule Precision@50:", round(precision_at_k(hand_rule_score, y, 50), 3))

Hand rule Precision@20: 0.9
Hand rule Precision@50: 0.68


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

We can claim an observed, decision-support ranking of pages to review first, backed by
transparent reason codes (stale_visible_page, declining_with_demand, low_ctr_visible_page,etc.), validated with client-holdout precision@K.
But we can't claim is a flagged page is guaranteed to recover if refreshed (that needs a causal experiment, not this data).This reproduces or predicts Google's algorithm, being that the current is_declining_label (a same-window proxy) equals a true future outcome — a stronger version of this lane would define the label as a future window (e.g. prior 90 days -> next 30 days decline) rather than the current-window proxy used here.<br>
**The model supports a human reviewer's decision — it does not replace it**.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.